# Programa 1

In [ ]:
%pip install wordcloud matplotlib ipywidgets pillow numpy

In [1]:
import io
import base64
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from PIL import Image, ImageOps
from wordcloud import WordCloud

LARGURA, ALTURA = 800, 400
FICHEIRO_SAIDA = "nuvem_palavras.png"
PALETAS = ["GnBu", "plasma", "Blues", "viridis", "Greens", "cool",
           "winter", "YlGnBu", "BuGn", "ocean", "flag"]

# ------------------------------- Widgets (interface) -------------------------------

caixa_texto = widgets.Textarea(
    value="Estar em toda a parte onde houver bem a fazer e sofrimento a aliviar",
    placeholder="Cola aqui o teu texto...",
    description="Texto:",
    layout=widgets.Layout(width="100%", height="140px"),
    style={"description_width": "110px"},
)

# Cada palavra escrita + Enter vira uma "tag" (palavra a remover da nuvem)
tags_stopwords = widgets.TagsInput(
    value=["em", "a", "toda", "onde", "e"],
    allow_duplicates=False,
    description="Palavras a ignorar:",
    style={"description_width": "110px"},
)

paleta = widgets.Dropdown(
    options=PALETAS, value="GnBu",
    description="Paleta de cores:",
    style={"description_width": "110px"},
)

cor_fundo = widgets.ColorPicker(
    value="#DDEEE7",
    description="Cor de fundo:",
    style={"description_width": "110px"},
)

num_palavras = widgets.IntSlider(
    value=50, min=1, max=200, step=1,
    description="Nº de palavras:",
    continuous_update=False,
    style={"description_width": "110px"},
)

modo_imagem = widgets.Dropdown(
    options=[("Sem imagem", "nenhuma"),
             ("Imagem como fundo", "fundo"),
             ("Imagem como forma (máscara)", "mascara")],
    value="nenhuma",
    description="Uso da imagem:",
    style={"description_width": "110px"},
)

upload = widgets.FileUpload(accept="image/*", multiple=False, description="Carregar imagem")

botao_gerar = widgets.Button(description="Gerar nuvem", button_style="success", icon="refresh")
link_download = widgets.HTML()
saida = widgets.Output()

# ------------------------------- Funções -------------------------------

def ler_imagem_carregada():
    """Devolve a imagem carregada (PIL, RGBA) ou None. Compatível com ipywidgets 7 e 8."""
    valor = upload.value
    if not valor:
        return None
    ficheiro = valor[0] if isinstance(valor, (tuple, list)) else list(valor.values())[0]
    return Image.open(io.BytesIO(bytes(ficheiro["content"]))).convert("RGBA")


def criar_nuvem(texto, stopwords, imagem, modo):
    """Constrói a nuvem e devolve uma imagem PIL pronta a mostrar/guardar."""
    params = dict(
        width=LARGURA, height=ALTURA,
        colormap=paleta.value,
        max_words=num_palavras.value,
        stopwords=stopwords,
    )

    if modo == "mascara" and imagem is not None:
        # Fundo transparente passa a branco (o WordCloud ignora as zonas brancas)
        base = Image.new("RGBA", imagem.size, "white")
        base.alpha_composite(imagem)
        params["mask"] = np.array(base.convert("RGB"))
        params["background_color"] = cor_fundo.value
        return WordCloud(**params).generate(texto).to_image()

    if modo == "fundo" and imagem is not None:
        params["mode"] = "RGBA"
        params["background_color"] = None  # nuvem transparente
        nuvem = WordCloud(**params).generate(texto).to_image()
        fundo = ImageOps.fit(imagem, (LARGURA, ALTURA))  # ajusta sem deformar
        fundo.alpha_composite(nuvem)
        return fundo.convert("RGB")

    params["background_color"] = cor_fundo.value
    return WordCloud(**params).generate(texto).to_image()


def gerar(_=None):
    saida.clear_output(wait=True)
    link_download.value = ""
    texto = caixa_texto.value.strip()

    with saida:
        if not texto:
            print("Escreve ou cola um texto primeiro.")
            return
        stopwords = {t.lower() for t in tags_stopwords.value}
        try:
            img = criar_nuvem(texto, stopwords, ler_imagem_carregada(), modo_imagem.value)
        except ValueError:
            print("Não sobraram palavras para a nuvem (todas foram ignoradas?).")
            return

        plt.figure(figsize=(10, 5))
        plt.imshow(img, interpolation="bilinear")
        plt.axis("off")
        plt.show()

    img.save(FICHEIRO_SAIDA)  # cópia na pasta do notebook

    # Botão de download para o computador do utilizador
    buffer = io.BytesIO()
    img.save(buffer, format="PNG")
    b64 = base64.b64encode(buffer.getvalue()).decode()
    link_download.value = (
        f'<a download="{FICHEIRO_SAIDA}" href="data:image/png;base64,{b64}" '
        f'style="padding:8px 14px;background:#1f6feb;color:white;'
        f'border-radius:6px;text-decoration:none;">⬇ Descarregar PNG</a>'
    )


botao_gerar.on_click(gerar)

# ------------------------------- Layout -------------------------------

display(widgets.VBox([
    caixa_texto,
    tags_stopwords,
    widgets.HBox([paleta, cor_fundo]),
    num_palavras,
    widgets.HBox([upload, modo_imagem]),
    widgets.HBox([botao_gerar, link_download]),
    saida,
]))

gerar()  # mostra logo uma nuvem com o texto de exemplo
